# Sentiment Analysis Methods Compared

This notebook compares four approaches to financial sentiment analysis:

1. **Bag of Words** — count positive and negative words from a general list
2. **Loughran-McDonald Dictionary** — finance-specific word lists
3. **FinBERT** — a neural network fine-tuned on financial text
4. **LLM (Gemma 4 31B)** — a large language model via OpenRouter

Change `sentence = s1` to `sentence = s2`, etc., to see how each method handles different challenges.

## Install Libraries

Run this cell once. On Colab it takes about a minute.

In [ ]:
!pip install -q transformers torch openai

## OpenRouter API Key

Method 4 (LLM) requires a free OpenRouter API key.

1. Create a free account at [openrouter.ai](https://openrouter.ai/).
2. Go to [openrouter.ai/keys](https://openrouter.ai/keys) and create a key.
3. Add credit (even $0 works for free models) if prompted.

**Option A — Colab Secrets (recommended):**
1. Click the 🔑 key icon in the left sidebar of Colab.
2. Click **Add new secret**. Name it `OPENROUTER_API_KEY` and paste your key.
3. Toggle "Notebook access" on.

**Option B — Paste directly:**
Replace `None` in the cell below with your key as a string, e.g., `"sk-or-..."`.

In [ ]:
# Option B: paste your key here if not using Colab Secrets
openrouter_key = None  # e.g., "sk-or-..."

# Try Colab Secrets first
try:
    from google.colab import userdata
    openrouter_key = userdata.get("OPENROUTER_API_KEY")
except Exception:
    pass

if openrouter_key is None:
    print("⚠ No OpenRouter key found. Methods 1-3 will still work.")
    print("  To use Method 4 (LLM), add OPENROUTER_API_KEY to Colab Secrets")
    print("  or set openrouter_key = 'sk-or-...' above.")
else:
    print("✓ OpenRouter key loaded.")

## Test Sentences

Each sentence illustrates a different challenge for sentiment analysis.

In [ ]:
# Easy case — all methods should get this right
s1 = "Apple revenue beats estimates, shares surge 5%"

# Contrast/hedging — 'despite' dismisses the positive
s2 = "Despite strong revenue, the company warned about supply chain disruptions"

# Negation — 'not' reverses the sentiment
s3 = "The company did not report a loss"

# Relative framing — fell less than feared is actually good news
s4 = "Shares fell less than feared after earnings miss"

# Forward guidance matters more than trailing results
s5 = "Revenue growth was strong but management issued cautious guidance for next quarter"

# Finance-specific words that confuse general dictionaries
s6 = "The firm reduced its outstanding liabilities and increased capital reserves"

# Sarcasm / implicit sentiment
s7 = "Another quarter of record profits, yet the stock barely moved"

# Mixed with numbers
s8 = "Net income fell 12% as rising input costs squeezed margins"

## Choose a Sentence

Change `s1` below to any of `s1` through `s8`, then run the cells below.

In [ ]:
sentence = s1

## Method 1: Bag of Words

Count words from simple positive/negative lists. No understanding of word order, negation, or context.

In [ ]:
positive_words = {
    "good", "strong", "growth", "profit", "profits", "beat", "beats",
    "surge", "upgrade", "record", "increased", "gain", "gains",
    "positive", "rose", "improved", "recovery", "optimistic",
}
negative_words = {
    "bad", "weak", "loss", "decline", "miss", "plunge", "downgrade",
    "warned", "disruptions", "fell", "feared", "cautious", "risk",
    "liability", "liabilities", "volatile", "squeezed", "reduced",
}

words = sentence.lower().split()
pos = [w for w in words if w.strip(".,;:!?") in positive_words]
neg = [w for w in words if w.strip(".,;:!?") in negative_words]
score = len(pos) - len(neg)
label = "positive" if score > 0 else "negative" if score < 0 else "neutral"

print(f"Sentence: {sentence}")
print(f"Positive words found: {pos}")
print(f"Negative words found: {neg}")
print(f"Score: {score:+d} → {label}")

## Method 2: Loughran-McDonald Dictionary

Finance-specific word lists built from 10-K filings. Fixes the domain problem (e.g., 'liability' is not negative in finance) but still just counts words.

In [ ]:
# Subset of the Loughran-McDonald lists (the full lists have 2,355 negative and 354 positive words)
lm_positive = {
    "achieve", "attain", "beat", "beats", "benefit", "efficient",
    "favorable", "gain", "gains", "growth", "improved", "innovative",
    "profitable", "profit", "profits", "record", "stabilize", "strong",
    "surge", "surpass", "upturn", "recovery",
}
lm_negative = {
    "breach", "cautious", "decline", "default", "delinquent",
    "disruptions", "downgrade", "fell", "impairment", "insolvent",
    "litigation", "loss", "miss", "overdue", "penalized", "plunge",
    "restated", "squeezed", "underperform", "warned", "weak",
}

words = sentence.lower().split()
pos = [w for w in words if w.strip(".,;:!?") in lm_positive]
neg = [w for w in words if w.strip(".,;:!?") in lm_negative]
score = len(pos) - len(neg)
label = "positive" if score > 0 else "negative" if score < 0 else "neutral"

print(f"Sentence: {sentence}")
print(f"LM positive words found: {pos}")
print(f"LM negative words found: {neg}")
print(f"Score: {score:+d} → {label}")

## Method 3: FinBERT

A neural network fine-tuned on financial text. Understands context, negation, hedging, and word order. Returns probabilities for positive, negative, and neutral.

In [ ]:
import warnings
warnings.filterwarnings("ignore")
import logging
logging.disable(logging.WARNING)

from transformers import pipeline

finbert = pipeline("text-classification", model="ProsusAI/finbert", top_k=None)

results = finbert(sentence)[0]
results.sort(key=lambda x: x["score"], reverse=True)
best = results[0]

print(f"Sentence: {sentence}")
print()
for r in results:
    bar = "█" * int(r["score"] * 40)
    print(f"  {r['label']:>8s}: {r['score']:5.1%} {bar}")
print(f"\nVerdict: {best['label']} ({best['score']:.0%})")

## Method 4: LLM (Gemma 4 31B via OpenRouter)

A large language model that can reason about sentiment with zero training data — just a prompt. Uses the free Gemma 4 31B model through OpenRouter's OpenAI-compatible API.

This requires an OpenRouter API key (see setup above).

In [ ]:
from openai import OpenAI

def llm_sentiment(text, api_key):
    """Classify sentiment using an LLM via OpenRouter."""
    client = OpenAI(
        base_url="https://openrouter.ai/api/v1",
        api_key=api_key,
    )
    response = client.chat.completions.create(
        model="google/gemma-4-31b-it:free",
        messages=[
            {"role": "system", "content": (
                "You are a financial sentiment classifier. "
                "Classify the sentiment of the given financial headline as "
                "positive, negative, or neutral. "
                "Reply with exactly one word (positive, negative, or neutral) "
                "followed by a brief explanation."
            )},
            {"role": "user", "content": text},
        ],
        temperature=0,
    )
    return response.choices[0].message.content

if openrouter_key:
    print(f"Sentence: {sentence}")
    print(f"\nGemma 4 31B says:\n")
    print(llm_sentiment(sentence, openrouter_key))
else:
    print("⚠ Skipped — no OpenRouter API key. See setup instructions above.")

## Run All Sentences at Once

Compare all four methods side by side.

In [ ]:
all_sentences = [s1, s2, s3, s4, s5, s6, s7, s8]

def bow_label(s, pos_set, neg_set):
    words = s.lower().split()
    score = sum(1 for w in words if w.strip(".,;:!?") in pos_set) - \
            sum(1 for w in words if w.strip(".,;:!?") in neg_set)
    return "positive" if score > 0 else "negative" if score < 0 else "neutral"

def llm_label(s, api_key):
    """Extract just the label from the LLM response."""
    try:
        resp = llm_sentiment(s, api_key)
        first_word = resp.strip().split()[0].lower().strip(".,;:!?")
        if first_word in ("positive", "negative", "neutral"):
            return first_word
        return resp[:20] + "..."
    except Exception as e:
        return f"error"

finbert_results = finbert([s for s in all_sentences])

has_llm = openrouter_key is not None
header = f"{'':>3s}  {'Bag of Words':>14s}  {'Loughran-McD':>14s}  {'FinBERT':>14s}"
if has_llm:
    header += f"  {'Gemma 4':>14s}"
header += "  Sentence"
print(header)
print("─" * (115 if has_llm else 100))

for i, (s, fb) in enumerate(zip(all_sentences, finbert_results), 1):
    bow = bow_label(s, positive_words, negative_words)
    lm = bow_label(s, lm_positive, lm_negative)
    best_fb = max(fb, key=lambda x: x["score"])
    fb_str = f"{best_fb['label']} ({best_fb['score']:.0%})"
    row = f"s{i}:  {bow:>14s}  {lm:>14s}  {fb_str:>14s}"
    if has_llm:
        llm_result = llm_label(s, openrouter_key)
        row += f"  {llm_result:>14s}"
    row += f"  {s}"
    print(row)